# Indian vehicle detection and Re-ID (Colab)

This notebook has three independent video workflows:

1. **Detect all vehicles** in uploaded videos.
2. Upload a **cropped reference vehicle** and search uploaded videos; matching detections get white boxes.
3. Do the same target search and also draw green boxes around all other detected vehicles.

**Before starting:** in Colab choose **Runtime → Change runtime type → T4 GPU**. Run the setup cell once, then use whichever processing cells you need. The setup loads the UVH-26 India-tuned YOLO model from Drive (or downloads it) and loads the VeRi-trained ResNet checkpoint if present. No Gradio app is used.

In [2]:
!pip -q install -U "ultralytics>=8.4" huggingface_hub timm

from pathlib import Path
from google.colab import drive, files
from huggingface_hub import hf_hub_download
from IPython.display import display, Video, FileLink, Markdown
from PIL import Image
from ultralytics import YOLO
from torchvision import models, transforms as T
from torch import nn
import torch
import torch.nn.functional as F
import cv2, numpy as np, os, subprocess, tempfile, time
import timm

drive.mount("/content/drive")
assert torch.cuda.is_available(), "Select a T4 GPU runtime, then rerun this cell."
DEVICE = torch.device("cuda:0")
print("GPU:", torch.cuda.get_device_name(0))

PROJECT = Path("/content/drive/MyDrive/reid_project")
MODELS = PROJECT / "models"
YOLO_WEIGHTS = MODELS / "weights/YOLOv11-S/UVH-26-MV-YOLOv11-S.pt"
REID_WEIGHTS = MODELS / "fused_reid_veri.pt"
OUTPUT_DIR = Path("/content/vehicle_reid_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
YOLO_WEIGHTS.parent.mkdir(parents=True, exist_ok=True)

if not YOLO_WEIGHTS.is_file():
    hf_hub_download(
        repo_id="iisc-aim/UVH-26",
        repo_type="model",
        filename="weights/YOLOv11-S/UVH-26-MV-YOLOv11-S.pt",
        local_dir=str(MODELS),
    )
detector = YOLO(str(YOLO_WEIGHTS))
print("Detector:", YOLO_WEIGHTS)

import timm

class ResNetMidInfer(nn.Module):
    def __init__(self, last_stride=1):
        super().__init__()
        r = models.resnet50(weights=None)
        if last_stride == 1:
            r.layer4[0].conv2.stride = (1, 1)
            r.layer4[0].downsample[0].stride = (1, 1)
        self.stem = nn.Sequential(r.conv1, r.bn1, r.relu, r.maxpool, r.layer1, r.layer2)
        self.layer3, self.layer4 = r.layer3, r.layer4
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.bn = nn.BatchNorm1d(1024 + 2048)
    def forward(self, x):
        x = self.stem(x); mid = self.layer3(x); high = self.layer4(mid)
        return self.bn(torch.cat([self.pool(mid).flatten(1), self.pool(high).flatten(1)], 1))

class SwinInfer(nn.Module):
    def __init__(self, name):
        super().__init__()
        self.body = timm.create_model(name, pretrained=False, num_classes=0)
        self.bn = nn.BatchNorm1d(self.body.num_features)
    def forward(self, x):
        return self.bn(self.body(x))

if not REID_WEIGHTS.is_file():
    raise FileNotFoundError(f"Run the save cell in the training notebook first: {REID_WEIGHTS}")

_b = torch.load(REID_WEIGHTS, map_location="cpu")
res_model  = ResNetMidInfer(_b["last_stride"])
swin_model = SwinInfer(_b["swin_name"])
res_model.load_state_dict(_b["resnet"], strict=True)     # strict: any mismatch raises immediately
swin_model.load_state_dict(_b["swin"],  strict=True)
res_model, swin_model = res_model.to(DEVICE).eval(), swin_model.to(DEVICE).eval()
W_RES = _b["w_res"]
REID_LABEL = f"Fused ResNetmid + Swin-S (w_res={W_RES:.2f})"
print("Re-ID model:", REID_LABEL)

def _tf(size):
    return T.Compose([T.Resize(size), T.ToTensor(), T.Normalize(_b["mean"], _b["std"])])
RES_TF, SWIN_TF = _tf(_b["res_size"]), _tf(_b["swin_size"])

USE_FP16 = True
FRAME_BATCH = 4        # lower to 1-2 if the GPU runs out of memory
FRAME_STRIDE = 1       # analyze every frame; raise to 2-3 for speed
MATCH_THRESHOLD = 0.70 # starting value only, calibrate below

@torch.inference_mode()
def embed_rgb_images(rgb_images, chunk_size=16, flip=False):
    """Returns unit-norm fused embeddings (N, 3840). Cosine = w*sim_resnet + (1-w)*sim_swin."""
    out = []
    for s in range(0, len(rgb_images), chunk_size):
        pils = [Image.fromarray(np.asarray(im, dtype=np.uint8)) for im in rgb_images[s:s + chunk_size]]
        xr = torch.stack([RES_TF(p)  for p in pils]).to(DEVICE)
        xs = torch.stack([SWIN_TF(p) for p in pils]).to(DEVICE)
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_FP16):
            fr, fs = res_model(xr).float(), swin_model(xs).float()
            if flip:
                fr = fr + res_model(torch.flip(xr, dims=[3])).float()
                fs = fs + swin_model(torch.flip(xs, dims=[3])).float()
        fr, fs = F.normalize(fr, dim=1), F.normalize(fs, dim=1)
        out.append(torch.cat([W_RES ** 0.5 * fr, (1 - W_RES) ** 0.5 * fs], dim=1).cpu())
    return torch.cat(out).numpy()

def _write_video_chunk(frames, frame_ids, writer, target_embedding, draw_all,
                       threshold, frame_stride, stats):
    sample_pos = [i for i, fid in enumerate(frame_ids) if fid % frame_stride == 0]
    results = []
    if sample_pos:
        results = detector.predict(
            [frames[i] for i in sample_pos],
            imgsz=640,
            conf=0.15,
            batch=min(FRAME_BATCH, len(sample_pos)),
            device=0,
            quantize=16 if USE_FP16 else None,
            verbose=False,
        )

    by_pos = {pos: [] for pos in sample_pos}
    crop_refs, crops_rgb = [], []
    for pos, result in zip(sample_pos, results):
        boxes = result.boxes
        if boxes is None:
            continue
        for box in boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
            h, w = frames[pos].shape[:2]
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(w, x2), min(h, y2)
            if x2 <= x1 or y2 <= y1:
                continue
            cls_id = int(box.cls[0].item())
            conf = float(box.conf[0].item())
            class_names = detector.names
            class_name = class_names.get(cls_id, str(cls_id)) if isinstance(class_names, dict) else class_names[cls_id]
            det_index = len(by_pos[pos])
            by_pos[pos].append({"box": (x1, y1, x2, y2), "cls": class_name,
                                "conf": conf, "sim": None})
            stats["boxes"] += 1
            if target_embedding is not None:
                crop = frames[pos][y1:y2, x1:x2]
                crops_rgb.append(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
                crop_refs.append((pos, det_index))

    if crops_rgb:
        similarities = embed_rgb_images(crops_rgb) @ target_embedding
        for (pos, det_index), sim in zip(crop_refs, similarities):
            det = by_pos[pos][det_index]
            det["sim"] = float(sim)
            stats["best"] = max(stats["best"], float(sim))
            stats["hits"] += int(float(sim) >= threshold)

    for pos, frame in enumerate(frames):
        for det in by_pos.get(pos, []):
            sim = det["sim"]
            matched = sim is not None and sim >= threshold
            if matched:
                color, thickness = (255, 255, 255), 3
                label = f"TARGET {sim:.2f}"
            elif draw_all:
                color, thickness = (0, 220, 80), 2
                label = f"{det['cls']} {det['conf']:.2f}"
            else:
                continue
            x1, y1, x2, y2 = det["box"]
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)
            cv2.putText(frame, label, (x1, max(22, y1 - 7)), cv2.FONT_HERSHEY_SIMPLEX,
                        0.65, color, 2, cv2.LINE_AA)
        writer.write(frame)

def process_video(video_path, target_embedding=None, draw_all=True,
                  threshold=MATCH_THRESHOLD, frame_stride=FRAME_STRIDE):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    if width < 1 or height < 1:
        cap.release()
        raise RuntimeError(f"Could not read dimensions for: {video_path}")

    stem = Path(video_path).stem
    raw_path = OUTPUT_DIR / f"{stem}_raw.mp4"
    output_path = OUTPUT_DIR / f"{stem}_annotated.mp4"
    writer = cv2.VideoWriter(str(raw_path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))
    if not writer.isOpened():
        cap.release()
        raise RuntimeError("Could not create the video writer.")

    stats = {"boxes": 0, "hits": 0, "best": -1.0}
    frames, frame_ids = [], []
    frame_count, t0 = 0, time.perf_counter()
    target = None if target_embedding is None else np.asarray(target_embedding, dtype=np.float32)
    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            frames.append(frame)
            frame_ids.append(frame_count)
            frame_count += 1
            if len(frames) >= FRAME_BATCH:
                _write_video_chunk(frames, frame_ids, writer, target, draw_all,
                                   threshold, frame_stride, stats)
                frames, frame_ids = [], []
        if frames:
            _write_video_chunk(frames, frame_ids, writer, target, draw_all,
                               threshold, frame_stride, stats)
    finally:
        cap.release()
        writer.release()
    if frame_count == 0:
        raise RuntimeError(f"No frames found in: {video_path}")

    if subprocess.run(["ffmpeg", "-version"], capture_output=True).returncode == 0:
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(raw_path),
                        "-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "23",
                        str(output_path)], check=True)
        raw_path.unlink(missing_ok=True)
    else:
        raw_path.replace(output_path)

    rate = frame_count / max(time.perf_counter() - t0, 1e-6)
    if target is None:
        status = f"DETECTION ONLY | {stats['boxes']} boxes | {rate:.1f} processed FPS"
    elif stats["hits"] == 0:
        status = f"MISSING | {frame_count} frames | {rate:.1f} processed FPS"
    else:
        status = f"FOUND | best cosine {stats['best']:.3f} | {stats['hits']} hits | {rate:.1f} processed FPS"
    return output_path, status

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: Tesla T4
Detector: /content/drive/MyDrive/reid_project/models/weights/YOLOv11-S/UVH-26-MV-YOLOv11-S.pt
Re-ID model: Fused ResNetmid + Swin-S (w_res=0.00)


## Cell 1 — detect all vehicles

Run this cell, select one or more video files in the upload dialog, and wait for each annotated result. Green boxes include the UVH-26 vehicle class and detector confidence.

In [3]:
print("Select all videos for vehicle detection.")
uploaded = files.upload()
video_paths = [Path(name) for name in uploaded if Path(name).suffix.lower() in {".mp4", ".avi", ".mov", ".mkv"}]
if not video_paths:
    raise ValueError("No video files were uploaded.")

detection_outputs = []
for path in video_paths:
    output_path, status = process_video(path, target_embedding=None, draw_all=True)
    detection_outputs.append(output_path)
    display(Markdown(f"### {path.name} — {status}"))
    display(Video(str(output_path), embed=True, width=800))
    display(FileLink(str(output_path)))

Select all videos for vehicle detection.


Saving vidssave.com HOW TRIPLE RIDING VIOLATION _ Caught By CCTV Cam _ Tirupati Traffic Police 360P.mp4 to vidssave.com HOW TRIPLE RIDING VIOLATION _ Caught By CCTV Cam _ Tirupati Traffic Police 360P.mp4
Saving vidssave.com Indian Traffic, Vehicles, Highway Footage for Object Detection 360P.mp4 to vidssave.com Indian Traffic, Vehicles, Highway Footage for Object Detection 360P.mp4


### vidssave.com HOW TRIPLE RIDING VIOLATION _ Caught By CCTV Cam _ Tirupati Traffic Police 360P.mp4 — DETECTION ONLY | 7957 boxes | 28.8 processed FPS

/content/vehicle_reid_outputs/vidssave.com HOW TRIPLE RIDING VIOLATION _ Caught By CCTV Cam _ Tirupati Traffic Police 360P_annotated.mp4

### vidssave.com Indian Traffic, Vehicles, Highway Footage for Object Detection 360P.mp4 — DETECTION ONLY | 22223 boxes | 24.0 processed FPS

/content/vehicle_reid_outputs/vidssave.com Indian Traffic, Vehicles, Highway Footage for Object Detection 360P_annotated.mp4

## Cell 2 — find the cropped reference vehicle

Upload one cropped vehicle image, then upload the videos to search. Only target matches are outlined in white. A result marked **MISSING** means no detection exceeded the current similarity threshold.

In [4]:
print("Upload one cropped reference image.")
reference_upload = files.upload()
image_paths = [Path(name) for name in reference_upload
               if Path(name).suffix.lower() in {".png", ".jpg", ".jpeg", ".webp"}]
if len(image_paths) != 1:
    raise ValueError("Upload exactly one cropped reference image in this prompt.")
ref_rgb = np.asarray(Image.open(image_paths[0]).convert("RGB"))
target_embedding = embed_rgb_images([ref_rgb])[0]
display(Markdown(f"Reference crop loaded: **{image_paths[0].name}**. Re-ID model: **{REID_LABEL}**."))

print("Now upload all videos to search for that vehicle.")
uploaded = files.upload()
video_paths = [Path(name) for name in uploaded if Path(name).suffix.lower() in {".mp4", ".avi", ".mov", ".mkv"}]
if not video_paths:
    raise ValueError("No video files were uploaded.")

target_outputs = []
for path in video_paths:
    output_path, status = process_video(path, target_embedding=target_embedding, draw_all=False)
    target_outputs.append(output_path)
    display(Markdown(f"### {path.name} — {status}"))
    display(Video(str(output_path), embed=True, width=800))
    display(FileLink(str(output_path)))

Upload one cropped reference image.


Saving Screenshot 2026-11-19 122606.png to Screenshot 2026-11-19 122606.png


Reference crop loaded: **Screenshot 2026-11-19 122606.png**. Re-ID model: **Fused ResNetmid + Swin-S (w_res=0.00)**.

Now upload all videos to search for that vehicle.


Saving vidssave.com HOW TRIPLE RIDING VIOLATION _ Caught By CCTV Cam _ Tirupati Traffic Police 360P.mp4 to vidssave.com HOW TRIPLE RIDING VIOLATION _ Caught By CCTV Cam _ Tirupati Traffic Police 360P (1).mp4
Saving vidssave.com Indian Traffic, Vehicles, Highway Footage for Object Detection 360P.mp4 to vidssave.com Indian Traffic, Vehicles, Highway Footage for Object Detection 360P (1).mp4


### vidssave.com HOW TRIPLE RIDING VIOLATION _ Caught By CCTV Cam _ Tirupati Traffic Police 360P (1).mp4 — MISSING | 805 frames | 8.8 processed FPS

/content/vehicle_reid_outputs/vidssave.com HOW TRIPLE RIDING VIOLATION _ Caught By CCTV Cam _ Tirupati Traffic Police 360P (1)_annotated.mp4

### vidssave.com Indian Traffic, Vehicles, Highway Footage for Object Detection 360P (1).mp4 — FOUND | best cosine 0.902 | 101 hits | 3.5 processed FPS

/content/vehicle_reid_outputs/vidssave.com Indian Traffic, Vehicles, Highway Footage for Object Detection 360P (1)_annotated.mp4

## Cell 3 — find the target and detect all vehicles

This repeats the reference-image and video upload flow. Matching target detections get white boxes; other detected vehicles get green boxes with class labels.

In [5]:
print("Upload one cropped reference image.")
reference_upload = files.upload()
image_paths = [Path(name) for name in reference_upload
               if Path(name).suffix.lower() in {".png", ".jpg", ".jpeg", ".webp"}]
if len(image_paths) != 1:
    raise ValueError("Upload exactly one cropped reference image in this prompt.")
ref_rgb = np.asarray(Image.open(image_paths[0]).convert("RGB"))
target_embedding = embed_rgb_images([ref_rgb])[0]
display(Markdown(f"Reference crop loaded: **{image_paths[0].name}**. Re-ID model: **{REID_LABEL}**."))

print("Now upload all videos to search and annotate.")
uploaded = files.upload()
video_paths = [Path(name) for name in uploaded if Path(name).suffix.lower() in {".mp4", ".avi", ".mov", ".mkv"}]
if not video_paths:
    raise ValueError("No video files were uploaded.")

target_and_detection_outputs = []
for path in video_paths:
    output_path, status = process_video(path, target_embedding=target_embedding, draw_all=True)
    target_and_detection_outputs.append(output_path)
    display(Markdown(f"### {path.name} — {status}"))
    display(Video(str(output_path), embed=True, width=800))
    display(FileLink(str(output_path)))

Upload one cropped reference image.


Saving Screenshot 2026-11-19 122606.png to Screenshot 2026-11-19 122606 (1).png


Reference crop loaded: **Screenshot 2026-11-19 122606 (1).png**. Re-ID model: **Fused ResNetmid + Swin-S (w_res=0.00)**.

Now upload all videos to search and annotate.


Saving vidssave.com HOW TRIPLE RIDING VIOLATION _ Caught By CCTV Cam _ Tirupati Traffic Police 360P.mp4 to vidssave.com HOW TRIPLE RIDING VIOLATION _ Caught By CCTV Cam _ Tirupati Traffic Police 360P (2).mp4
Saving vidssave.com Indian Traffic, Vehicles, Highway Footage for Object Detection 360P.mp4 to vidssave.com Indian Traffic, Vehicles, Highway Footage for Object Detection 360P (2).mp4


### vidssave.com HOW TRIPLE RIDING VIOLATION _ Caught By CCTV Cam _ Tirupati Traffic Police 360P (2).mp4 — MISSING | 805 frames | 8.4 processed FPS

/content/vehicle_reid_outputs/vidssave.com HOW TRIPLE RIDING VIOLATION _ Caught By CCTV Cam _ Tirupati Traffic Police 360P (2)_annotated.mp4

### vidssave.com Indian Traffic, Vehicles, Highway Footage for Object Detection 360P (2).mp4 — FOUND | best cosine 0.902 | 101 hits | 3.3 processed FPS

/content/vehicle_reid_outputs/vidssave.com Indian Traffic, Vehicles, Highway Footage for Object Detection 360P (2)_annotated.mp4